## Model specification

Let

- $y$ be a binary dependent variable
- $(x_1,\ldots,x_k)$ be explanatory variables or regressors

Denote

$$
\begin{aligned}
	x^\top&=\begin{pmatrix} 1 & x_1 & \cdots & x_k\end{pmatrix} &
	\beta^\top&=\begin{pmatrix} \beta_0 & \beta_1 & \cdots & \beta_k\end{pmatrix} &
\end{aligned}
$$

Then

$$
	x^\top\beta=\beta_0+\beta_1x_1+\cdots+\beta_kx_k
$$

Linear specification with a binary dependent variable

$$
	y=x^\top\beta+u
$$

Alternative specification

$$
	P(y=1)=x^\top\beta
$$

In [ ]:

import numpy as np
import pandas as pd

from statsmodels.formula.api import ols

from statsmodels.iolib.summary2 import summary_params, summary_col # output of test results

# critical values
from scipy.stats import t, f

# Do not show Warning
import warnings
warnings.simplefilter(action='ignore', category=Warning)
# Do not show ValueWarning, ConvergenceWarning from statsmodels
from statsmodels.tools.sm_exceptions import ValueWarning, ConvergenceWarning
warnings.simplefilter('ignore', category=ValueWarning)
warnings.simplefilter('ignore', category=ConvergenceWarning)

from IPython.display import Markdown, display

# load datasets as a dictionary
path_to_datasets = '../../datasets/econometrica-2/'
df_dict = {'loanapp': pd.read_csv(path_to_datasets+'loanapp.csv'), 
		'TableF5-1':pd.read_csv(path_to_datasets+'TableF5-1.csv'), 
		'SwissLabour': pd.read_csv(path_to_datasets+'SwissLabour.csv')}

## Model fitting

In [ ]:
df_names = ['loanapp', 'loanapp', 'TableF5-1', 'TableF5-1', 'SwissLabour']

endog_var = ['approve', 'approve', 'LFP', 'LFP', 'participation']

exog_vars =['mortno+unem+dep+male+married+yjob+self', 
			'appinc+I(appinc**2)+mortno+unem+dep+male+married+yjob+self',
			'WA+I(WA**2)+WE+KL6+K618+CIT+UN+np.log(FAMINC)',
			'WA+WE+CIT+UN+np.log(FAMINC)',
			'income+age+education+youngkids+oldkids']

mod_name = ['Approve equation No. 1', 'Approve equation No. 2', 'Labour force equation No. 1', 'Labour force equation No. 2', 'Swiss labour force equation No. 1']

digits = 3

for i, df_name in enumerate(df_names):
	# df = pd.read_csv('../datasets/'+df_name+'.csv')

	mod = ols(formula=endog_var[i]+'~'+exog_vars[i], data=df_dict[df_name])
	res = mod.fit()

	display(Markdown(f'### {mod_name[i]}'))
	display(Markdown(f'''For the dataset `{df_name}`, estimate the __LPM__ regression of `{endog_var[i]}` on `{exog_vars[i]}`
	and report its coefficients. __Round your answer to {digits} decimal places__ '''))

	df_coefs = pd.DataFrame({'Coefficient':res.params.index, 'Estimate':res.params.round(digits)})
	display(Markdown(f'''Answer:\n'''))
	display(Markdown(df_coefs.to_markdown(index=False)))
	# print(df_coefs.to_markdown(index=False))
	display(Markdown(f'''__Interpret the coefficients of the model__'''))

## t-test

In [ ]:
df_names = ['loanapp', 'loanapp', 'TableF5-1', 'TableF5-1', 'SwissLabour']

endog_var = ['approve', 'approve', 'LFP', 'LFP', 'participation']

exog_vars =['mortno+unem+dep+male+married+yjob+self', 
			'appinc+I(appinc**2)+mortno+unem+dep+male+married+yjob+self',
			'WA+I(WA**2)+WE+KL6+K618+CIT+UN+np.log(FAMINC)',
			'WA+WE+CIT+UN+np.log(FAMINC)',
			'income+I(income**2)+age+I(age**2)+education+youngkids+oldkids+foreign']

mod_name = ['Approve equation No. 1', 'Approve equation No. 2', 'Labour force equation No. 1', 'Labour force equation No. 2', 'Swiss labour force equation No. 1']

digits = 3

sign_level = [0.05, 0.01, 0.01, 0.05, 0.10]

for i, df_name in enumerate(df_names):
	# df = pd.read_csv('../datasets/'+df_name+'.csv')

	mod = ols(formula=endog_var[i]+'~'+exog_vars[i], data=df_dict[df_name])
	res = mod.fit(cov_type='HC3')

	display(Markdown(f'### {mod_name[i]}'))
	display(Markdown(f'''For the dataset `{df_name}`, estimate the __LPM__ regression of `{endog_var[i]}` on `{exog_vars[i]}`
	and test the significance of its coefficients. Significance level {sign_level[i]*100}%, number of observations {int(mod.nobs)}. 
	For each coefficient, report its test statistic, P-value,
	and draw a conclusion. Find the corresponding critical value. __Round your answer to {digits} decimal places__ '''))

	res_t_test = pd.DataFrame({'Coefficient':res.params.index, 
								'Test stat':res.tvalues.round(digits),
								'P-value':res.pvalues.round(digits),
								'Conclusion': res.pvalues.apply(lambda x: 'Significant' if x<sign_level[i] else 'Insignificant') })
	display(Markdown(f'''Answer:\n'''))
	display(Markdown(res_t_test.to_markdown(index=False)))
	# print(df_coefs.to_markdown(index=False))

	crit_val = t.isf(q=sign_level[i]/2, df=res.df_resid).round(digits)
	display(Markdown(f'''Critical value: {crit_val}'''))

## F-test: Significance of regression

For the regression

$$
	P(y=1)=x^\top\beta=\beta_0+\beta_1x_1+\cdots+\beta_kx_k
$$

test the hypothesis

$$
	H_0:\beta_1=\cdots=\beta_k=0
$$

In [ ]:
df_names = ['loanapp', 'loanapp', 'loanapp', 'SwissLabour']

endog_var = ['approve', 'approve', 'approve', 'participation']

exog_vars =['unem+male+yjob+self', 
			'appinc+I(appinc**2)+mortno+dep',
			'dep+male+married',
			'income+I(income**2)+education']

mod_name = ['Approve equation No. 1', 'Approve equation No. 2', 'Approve equation No. 3', 'Swiss labour force equation No. 1']

digits = 3

sign_level = [0.10, 0.05, 0.01, 0.05]

for i, df_name in enumerate(df_names):
	# df = pd.read_csv('../datasets/'+df_name+'.csv')

	mod = ols(formula=endog_var[i]+'~'+exog_vars[i], data=df_dict[df_name])
	res = mod.fit(cov_type='HC3')

	display(Markdown(f'### {mod_name[i]}'))
	display(Markdown(f'''For the dataset `{df_name}`, estimate the __LPM__ regression of `{endog_var[i]}` on `{exog_vars[i]}`.
	Test the overall significance of the regression. Significance level {sign_level[i]*100}%. 
	Find the value of the test statistic, its P-value, the critical value, and draw a conclusion.
	__Round your answer to {digits} decimal places__ '''))

	test_stat = res.fvalue
	p_val = res.f_pvalue
	crit_val = f.isf(sign_level[i], dfn=res.df_model, dfd=res.df_resid)
	is_significant = 'significant' if p_val<sign_level[i] else 'insignificant' 

	res_f_test_overall = pd.DataFrame({'Statistic': [round(test_stat, digits)], 
										'P-value': [round(p_val, digits)],
										'Critical': [crit_val.round(digits)], 
										'Regression': [is_significant]})

	display(Markdown(f'''Answer:\n'''))
	display(Markdown(res_f_test_overall.to_markdown(index=False)))
	# display(Markdown(f'Test statistic = {round(test_stat, digits)}'))
	# display(Markdown(f'P-value = {round(p_val, digits)}'))
	# display(Markdown(f'Critical value = {crit_val.round(digits)}'))
	# display(Markdown(f'Conclusion: the regression is __{is_significant}__'))
	# print(df_coefs.to_markdown(index=False))

## F-test: Joint significance

In [ ]:
df_names = ['loanapp', 'SwissLabour']

endog_var = ['approve', 'participation']

exog_vars =['appinc+I(appinc**2)+mortno+unem+dep+male+married+yjob+self',
			'income+I(income**2)+age+I(age**2)+education+youngkids+oldkids+foreign']

mod_name = ['Approve equation No. 1', 'Swiss labour force equation No. 1']

digits = 3

sign_level = [[0.05, 0.01], [0.05, 0.10]]

joint_sign = [ ['appinc, appinc**2', 'unem, dep, married'], ['income, income**2', 'age, age**2'] ]
hypotheses = [ ['appinc=0, I(appinc ** 2)=0', 'unem=0, dep=0, married=0'], 
				['income=0, I(income ** 2)=0', 'age=0, I(age ** 2)=0'] ]

for i, df_name in enumerate(df_names):
	# df = pd.read_csv('../datasets/'+df_name+'.csv')

	mod = ols(formula=endog_var[i]+'~'+exog_vars[i], data=df_dict[df_name])
	res = mod.fit()

	display(Markdown(f'### {mod_name[i]}'))
	display(Markdown(f'''For the dataset `{df_name}`, estimate the __LPM__ regression of `{endog_var[i]}` on `{exog_vars[i]}`
	Test the hypotheses below. For each hypothesis, give an interpretation, find the test statistic, its P-value, the critical value, and draw a conclusion.
	__Round your answer to {digits} decimal places__ '''))

	for j, hypothesis in enumerate(hypotheses[i]):
		display(Markdown(f'#### Hypothesis No. {j+1}'))
		display(Markdown(f'Test the joint significance of `{joint_sign[i][j]}`. Significance level {sign_level[i][j]*100}%'))

		res_f_test = res.wald_test(hypothesis, use_f=True)

		test_stat = res_f_test.statistic
		p_val = res_f_test.pvalue
		crit_val = f.isf(sign_level[i][j], dfn = res_f_test.df_denom, dfd = res.df_resid)
		is_significant = 'significant' if p_val<sign_level[i][j] else 'in significant'

		res_f_test_joint = pd.DataFrame({'Statistic': [round(test_stat, digits)], 
										'P-value': [round(p_val, digits)],
										'Critical': [crit_val.round(digits)], 
										'Joint effect': [is_significant]})

		display(Markdown(f'''Answer:\n'''))
		display(Markdown(res_f_test_joint.to_markdown(index=False)))

## Prediction

In [ ]:
df_names = ['loanapp', 'TableF5-1', 'SwissLabour']

endog_var = ['approve', 'LFP', 'participation']

exog_vars =['mortno+unem+dep+married', 
			'WA+I(WA**2)+CIT+UN+np.log(FAMINC)',
			'income+I(income**2)+age+education+youngkids+oldkids' ]

mod_name = ['Approve equation', 'labour force equation', 'Swiss labour force equation']

digits = 3

new_data = [ pd.DataFrame( {'mortno':[1, 1, 0], 'unem':[3.2, 3.9, 1.8], 'dep':[0, 1, 0], 'married':[1, 0, 1]} ), 
			pd.DataFrame( {'WA':[34, 36, 42], 'CIT':[1, 0, 0], 'UN':[3, 5, 7.5], 'FAMINC':[35000, 48500, 67800]} ),
			pd.DataFrame( {'income':[11.9, 10.49, 11.02], 'age':[3.2, 2.0, 4.4], 'education':[8, 19, 12], 'youngkids':[1, 0, 2], 'oldkids':[0,0,1]} ) ]

threshold = [0.5, 0.5]

for i, df_name in enumerate(df_names):
	# df = pd.read_csv('../datasets/'+df_name+'.csv')

	mod = ols(formula=endog_var[i]+'~'+exog_vars[i], data=df_dict[df_name])
	res = mod.fit()

	display(Markdown(f'### {mod_name[i]}'))
	display(Markdown(f'''For the dataset `{df_name}`, estimate the __LPM__ regression of `{endog_var[i]}` on `{exog_vars[i]}`. Consider the new data'''))
	display(Markdown(new_data[i].to_markdown(index=False)))
	display(Markdown(f'''Calculate a prediction for each observation and give an interpretation. __Round your answer to {digits} decimal places__ '''))

	y_pred_df = pd.DataFrame({'Prediction':res.predict(exog=new_data[i], transform=True).round(digits)})

	display(Markdown(f'''Answer:\n'''))
	display(Markdown(y_pred_df.to_markdown(index=False)))